In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

sns.set_theme(style='ticks', font_scale=1.1)

# Session-shift aPRF: how value tuning shifts between mappings

The session-shift model fits a LogGaussianPRF where the **mode** (preferred value)
shifts freely between sessions, while fwhm, amplitude, and baseline are shared.

If value representations adapt to the learned orientation-to-CHF mapping, we expect:
- **cdf** sessions (bimodal value distribution, peaks ~12 and ~32 CHF): modes cluster near those peaks
- **inverse_cdf** sessions (unimodal, peak ~22 CHF): modes cluster near 22 CHF

In [ ]:
R2_THR = 0.05
VALUE_MIN, VALUE_MAX = 2.5, 41.5
N_VOXELS_FI = 100

MAPPING_PALETTE = {'cdf': 'steelblue', 'inverse_cdf': 'tomato'}

# Value distribution peaks in CHF
MAPPING_PEAKS = {
    'cdf':         [12.0, 32.0],   # bimodal
    'inverse_cdf': [22.0],         # unimodal
}

# Auto-discover subjects with session-shift results
SS_DIR = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf-session-shift'
SUBJECTS = sorted([
    d.name.removeprefix('sub-')
    for d in SS_DIR.iterdir()
    if d.is_dir() and d.name.startswith('sub-')
]) if SS_DIR.exists() else []
print(f'Subjects: {SUBJECTS}')


def get_mapping(subject, session):
    num = int(''.join(c for c in str(subject) if c.isdigit()))
    if num % 2 == 0:
        return 'cdf' if session == 1 else 'inverse_cdf'
    return 'inverse_cdf' if session == 1 else 'cdf'

for s in SUBJECTS:
    print(f'  {s}: ses-1={get_mapping(s,1)}, ses-2={get_mapping(s,2)}')

In [ ]:
# ── load session-shift parameters within NPCr ─────────────────────────────
data = {}

for subject in SUBJECTS:
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    out_dir = SS_DIR / f'sub-{subject}' / 'func'

    def _path(desc):
        return out_dir / f'sub-{subject}_task-abstractvalue_space-T1w_desc-{desc}_pe.nii.gz'

    if not _path('r2').exists():
        print(f'sub-{subject}: no session-shift results, skipping')
        continue

    try:
        mask = sub.get_roi_mask('NPCr', hemi=None)
    except FileNotFoundError:
        print(f'sub-{subject}: no NPCr mask, skipping')
        continue

    # Use target_affine from the parameter maps to avoid resampling warnings
    from nilearn import image as nli
    ref = nli.load_img(str(_path('r2')))
    masker = NiftiMasker(mask_img=mask,
                         target_affine=ref.affine,
                         target_shape=ref.shape[:3]).fit()

    r2    = masker.transform(_path('r2')).squeeze()
    mode1 = masker.transform(_path('mode_1')).squeeze()
    mode2 = masker.transform(_path('mode_2')).squeeze()
    fwhm  = masker.transform(_path('fwhm')).squeeze()
    amp   = masker.transform(_path('amplitude')).squeeze()

    # Filter: R2 threshold + valid parameters
    sel = (r2 > R2_THR) & (fwhm > 0) & (amp != 0) & (mode1 > 0) & (mode2 > 0)
    print(f'sub-{subject}: {sel.sum():,} / {len(r2):,} voxels '
          f'(R2>{R2_THR}, valid params)')

    data[subject] = dict(
        r2=r2, mode1=mode1, mode2=mode2, fwhm=fwhm, amp=amp, sel=sel,
        map1=get_mapping(subject, 1),
        map2=get_mapping(subject, 2),
    )

In [ ]:
# ── joint mode plot (hexbin): mode_1 vs mode_2 ────────────────────────────
if data:
    subjects = sorted(data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 5.5),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        d = data[subject]
        m1, m2 = d['mode1'][d['sel']], d['mode2'][d['sel']]
        map1, map2 = d['map1'], d['map2']

        hb = ax.hexbin(m1, m2, gridsize=40, cmap='viridis',
                       extent=[VALUE_MIN, VALUE_MAX, VALUE_MIN, VALUE_MAX],
                       mincnt=1)

        # Identity line
        ax.plot([VALUE_MIN, VALUE_MAX], [VALUE_MIN, VALUE_MAX],
                'w--', lw=1, alpha=0.5)

        # Distribution peaks
        for px in MAPPING_PEAKS[map1]:
            ax.axvline(px, color='cyan', lw=1.5, ls=':', alpha=0.8)
        for py in MAPPING_PEAKS[map2]:
            ax.axhline(py, color='orange', lw=1.5, ls='--', alpha=0.8)

        ax.set_xlabel(f'Mode ses-1 ({map1}) [CHF]')
        ax.set_ylabel(f'Mode ses-2 ({map2}) [CHF]')
        ax.set_title(f'sub-{subject}  (n={d["sel"].sum():,})')
        ax.set_xlim(VALUE_MIN, VALUE_MAX)
        ax.set_ylim(VALUE_MIN, VALUE_MAX)
        ax.set_aspect('equal')
        plt.colorbar(hb, ax=ax, label='voxel count', shrink=0.8)

    fig.suptitle(f'Session-shift aPRF: preferred value per session (NPCr, R²>{R2_THR})',
                 fontsize=12)
    plt.show()

In [ ]:
# ── delta-mode histograms ──────────────────────────────────────────────────
if data:
    subjects = sorted(data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 4),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        d = data[subject]
        delta = d['mode2'][d['sel']] - d['mode1'][d['sel']]
        map1, map2 = d['map1'], d['map2']

        ax.hist(delta, bins=60, color='steelblue', edgecolor='none', alpha=0.8)
        ax.axvline(0, color='k', lw=1.5, ls='--')
        ax.axvline(delta.mean(), color='tomato', lw=2,
                   label=f'mean={delta.mean():.1f} CHF')
        ax.axvline(np.median(delta), color='orange', lw=2, ls=':',
                   label=f'median={np.median(delta):.1f} CHF')

        ax.set_xlabel(f'mode ses-2 ({map2}) - mode ses-1 ({map1}) [CHF]')
        ax.set_ylabel('voxel count')
        ax.set_title(f'sub-{subject}')
        ax.legend(fontsize=9)

    fig.suptitle('Shift in preferred value between sessions', fontsize=12)
    sns.despine()
    plt.show()

In [ ]:
# ── Fisher information per session (session-shift model) ──────────────────
if data:
    subjects = sorted(data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 4),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        found_any = False
        for ses in [1, 2]:
            mapping = get_mapping(subject, ses)
            color = MAPPING_PALETTE[mapping]
            fn = (SS_DIR / f'sub-{subject}' / f'ses-{ses}' / 'func'
                  / f'sub-{subject}_ses-{ses}_task-abstractvalue'
                    f'_mask-NPCr_nvoxels-{N_VOXELS_FI}_desc-fisherinfo_pe.tsv')
            if not fn.exists():
                continue
            fi = pd.read_csv(fn, sep='\t', index_col=0).squeeze()
            if (fi == 0).all():
                print(f'  sub-{subject} ses-{ses}: all-zero FI, skipping')
                continue
            peak = fi.index[fi.values.argmax()]
            ax.plot(fi.index, fi.values, lw=2, color=color,
                    label=f'ses-{ses} ({mapping})  peak={peak:.0f} CHF')
            ax.axvline(peak, color=color, lw=1, ls='--', alpha=0.5)
            found_any = True

        if not found_any:
            ax.text(0.5, 0.5, 'No valid FI data', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')

        ax.set_xlabel('Value (CHF)')
        ax.set_ylabel('Fisher information')
        ax.set_title(f'sub-{subject}')
        ax.legend(fontsize=9)

    fig.suptitle(f'Session-shift aPRF Fisher information (NPCr, top-{N_VOXELS_FI})',
                 fontsize=12)
    sns.despine()
    plt.show()

In [ ]:
# ── mode shift vs R²: do better-fit voxels shift more consistently? ───────
if data:
    subjects = sorted(data.keys())
    fig, axes = plt.subplots(1, len(subjects),
                              figsize=(6 * len(subjects), 4),
                              squeeze=False, constrained_layout=True)

    for ax, subject in zip(axes[0], subjects):
        d = data[subject]
        s = d['sel']
        delta = d['mode2'][s] - d['mode1'][s]
        r2_sel = d['r2'][s]

        ax.scatter(r2_sel, delta, s=2, alpha=0.3, c='steelblue', linewidths=0)
        ax.axhline(0, color='k', lw=1, ls='--', alpha=0.5)

        # Running mean in R² bins
        bins = np.linspace(R2_THR, r2_sel.max(), 20)
        bin_idx = np.digitize(r2_sel, bins)
        for b in range(1, len(bins)):
            mask = bin_idx == b
            if mask.sum() > 5:
                ax.plot(bins[b-1], delta[mask].mean(), 'o',
                        color='tomato', markersize=5, zorder=5)

        ax.set_xlabel('R²')
        ax.set_ylabel(f'mode shift (ses-2 - ses-1) [CHF]')
        ax.set_title(f'sub-{subject}')

    fig.suptitle('Mode shift vs model fit quality', fontsize=12)
    sns.despine()
    plt.show()